In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import xgboost as xgb

# Settings
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Libraries imported successfully!")
print(f"XGBoost version: {xgb.__version__}")

## 1. Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('../../alzheimers_disease_data.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nTarget Distribution:")
print(df['Diagnosis'].value_counts())
print(f"\nClass Balance: {(df['Diagnosis'].value_counts(normalize=True) * 100).round(2).to_dict()}")

# Display first few rows
df.head()

## 2. Feature Selection

Using the same 32 features as other models (excluding PatientID).

In [ ]:
# Feature columns (32 features)
FEATURE_COLUMNS = [
    'Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI', 'Smoking',
    'AlcoholConsumption', 'PhysicalActivity', 'DietQuality', 'SleepQuality',
    'FamilyHistoryAlzheimers', 'CardiovascularDisease', 'Diabetes',
    'Depression', 'HeadInjury', 'Hypertension', 'SystolicBP', 'DiastolicBP',
    'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides',
    'MMSE', 'FunctionalAssessment', 'MemoryComplaints', 'BehavioralProblems',
    'ADL', 'Confusion', 'Disorientation', 'PersonalityChanges',
    'DifficultyCompletingTasks', 'Forgetfulness'
]

TARGET = 'Diagnosis'

# Prepare features and target
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET].copy()

print(f"Features: {len(FEATURE_COLUMNS)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nNo missing values: {X.isnull().sum().sum() == 0}")

## 3. Train-Test Split

In [ ]:
# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

## 4. XGBoost Training with Cross-Validation

Using stratified K-fold cross-validation to ensure robust performance.

In [ ]:
# XGBoost parameters
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'n_estimators': 500,
    'learning_rate': 0.015,
    'max_depth': 10,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.05,
    'reg_lambda': 0.05,
    'gamma': 0.01,
    'random_state': RANDOM_STATE,
    'verbosity': 0,
    'n_jobs': -1
}

# Cross-validation setup
N_SPLITS = 5
cv_scores = []
oof_predictions = np.zeros(len(y_train))
best_iterations = []

print("Starting XGBoost Cross-Validation...")
print(f"Training with {N_SPLITS}-fold cross-validation\n")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Create XGBoost model
    xgb_model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=50)
    
    # Train
    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    # Predict
    y_pred_val = xgb_model.predict(X_val)
    y_pred_proba_val = xgb_model.predict_proba(X_val)[:, 1]
    
    # Store OOF predictions
    oof_predictions[val_idx] = y_pred_proba_val
    
    # Calculate metrics
    fold_accuracy = accuracy_score(y_val, y_pred_val)
    fold_auc = roc_auc_score(y_val, y_pred_proba_val)
    cv_scores.append(fold_accuracy)
    best_iterations.append(xgb_model.best_iteration)
    
    print(f"Fold {fold+1}: Accuracy = {fold_accuracy:.4f}, AUC = {fold_auc:.4f}, Best iter: {xgb_model.best_iteration}")

print(f"\n{'='*70}")
print(f"Mean CV Accuracy: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")
print(f"{'='*70}")

# OOF score
oof_predictions_binary = (oof_predictions > 0.5).astype(int)
oof_accuracy = accuracy_score(y_train, oof_predictions_binary)
oof_auc = roc_auc_score(y_train, oof_predictions)
print(f"Out-of-fold Accuracy: {oof_accuracy:.4f}")
print(f"Out-of-fold AUC: {oof_auc:.4f}")

avg_best_iter = int(np.mean(best_iterations))
print(f"Average best iteration: {avg_best_iter}")

## 5. Train Final Model

Training final model on all training data with optimal number of estimators.

In [ ]:
# Train final model on all training data
print("\nTraining final XGBoost model on all training data...")

final_xgb_params = xgb_params.copy()
final_xgb_params['n_estimators'] = avg_best_iter

xgb_model_final = xgb.XGBClassifier(**final_xgb_params)
xgb_model_final.fit(X_train, y_train)

print(f"✓ Final XGBoost model trained with {avg_best_iter} estimators")

## 6. Model Evaluation

In [ ]:
# Predictions on test set
y_pred = xgb_model_final.predict(X_test)
y_pred_proba = xgb_model_final.predict_proba(X_test)[:, 1]
y_pred_train = xgb_model_final.predict(X_train)

# Calculate metrics
xgb_test_acc = accuracy_score(y_test, y_pred)
xgb_train_acc = accuracy_score(y_train, y_pred_train)
xgb_precision = precision_score(y_test, y_pred)
xgb_recall = recall_score(y_test, y_pred)
xgb_f1 = f1_score(y_test, y_pred)
xgb_auc = roc_auc_score(y_test, y_pred_proba)

print("="*70)
print("XGBOOST PERFORMANCE")
print("="*70)
print(f"Test Accuracy:  {xgb_test_acc:.4f}")
print(f"Train Accuracy: {xgb_train_acc:.4f}")
print(f"Precision:      {xgb_precision:.4f}")
print(f"Recall:         {xgb_recall:.4f}")
print(f"F1-Score:       {xgb_f1:.4f}")
print(f"ROC-AUC:        {xgb_auc:.4f}")
print(f"CV Accuracy:    {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Diagnosis', 'Diagnosis']))

## 7. Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title('Confusion Matrix - XGBoost', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks([0.5, 1.5], ['No Diagnosis', 'Diagnosis'])
plt.yticks([0.5, 1.5], ['No Diagnosis', 'Diagnosis'])
plt.tight_layout()
plt.show()

print("Confusion Matrix:")
print(cm)

## 8. ROC Curve

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='green', lw=2, label=f'ROC curve (AUC = {xgb_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - XGBoost', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Feature Importance Analysis

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': xgb_model_final.feature_importances_
}).sort_values('importance', ascending=False)

print("="*70)
print("TOP 15 MOST IMPORTANT FEATURES")
print("="*70)
print(importance_df.head(15).to_string(index=False))

# Plot feature importance
plt.figure(figsize=(10, 8))
top_features = importance_df.head(15)
plt.barh(range(len(top_features)), top_features['importance'], color='forestgreen')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance', fontsize=12)
plt.title('Top 15 Feature Importance - XGBoost', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Save Model and Artifacts

In [ ]:
# Save model
joblib.dump(xgb_model_final, 'xgboost_model.pkl')
print(f"✓ Model saved: xgboost_model.pkl")

# Save metrics
metrics = {
    'model': 'XGBoost',
    'accuracy': float(xgb_test_acc),
    'train_accuracy': float(xgb_train_acc),
    'precision': float(xgb_precision),
    'recall': float(xgb_recall),
    'f1_score': float(xgb_f1),
    'roc_auc': float(xgb_auc),
    'cv_mean': float(np.mean(cv_scores)),
    'cv_std': float(np.std(cv_scores)),
    'oof_accuracy': float(oof_accuracy),
    'oof_auc': float(oof_auc),
    'best_iteration': int(avg_best_iter),
    'parameters': final_xgb_params
}

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)
print(f"✓ Metrics saved: metrics.json")

# Save predictions
predictions_df = pd.DataFrame({
    'true_label': y_test.values,
    'predicted_label': y_pred,
    'probability_no_diagnosis': xgb_model_final.predict_proba(X_test)[:, 0],
    'probability_diagnosis': xgb_model_final.predict_proba(X_test)[:, 1]
})
predictions_df.to_csv('predictions.csv', index=False)
print(f"✓ Predictions saved: predictions.csv")

# Save feature importance
importance_df.to_csv('feature_importance.csv', index=False)
print(f"✓ Feature importance saved: feature_importance.csv")

# Save feature names
joblib.dump(FEATURE_COLUMNS, 'feature_names.pkl')
print(f"✓ Feature names saved: feature_names.pkl")

print("\n" + "="*70)
print("✅ XGBOOST TRAINING COMPLETE!")
print("="*70)